# Teaching MLflow: Experiment Tracking, Model Registry & Reproducibility

**Audience assumption:** you already know machine learning and how to train a
model. This notebook is *not* about ML concepts — it's about a problem every
ML practitioner runs into once they start training more than a handful of
models:

> "Which run gave me 94% accuracy again? What hyperparameters did I use? Where
> did I save that model file? Is this the same model that's in production?"

**MLflow** is an open-source platform that solves this. It has four main components:

| Component | What it does |
|---|---|
| **Tracking** | Logs parameters, metrics, and artifacts (plots, files, models) for every training run, so you can compare runs later |
| **Models** | A standard format for packaging models so they can be loaded/served consistently regardless of which library trained them |
| **Model Registry** | A central store for versioning models and managing their lifecycle (e.g. staging → production) |
| **Projects** | A format for packaging reproducible ML code (not covered in depth here) |

This notebook focuses on **Tracking**, **Models**, and the **Registry** — the
three pieces you'll use almost daily.

## Dataset

We'll use the **Wine** dataset (`sklearn.datasets.load_wine`) — a small,
publicly available, real-world dataset (UCI Machine Learning Repository,
shipped directly with scikit-learn) of 13 chemical measurements of wines from
3 different cultivars. It's a classification problem, trains in milliseconds,
and lets us focus entirely on MLflow rather than on the model itself.

## Setup (run this once, in a terminal, before starting the notebook)

```bash
pip install mlflow scikit-learn pandas matplotlib seaborn
```

We will *not* start the MLflow UI from inside the notebook — it's a
long-running web server, so it belongs in its own terminal:

```bash
# Run this in a terminal, from the same directory as this notebook
mlflow ui --backend-store-uri ./mlruns
```

Then open **http://127.0.0.1:5000** in a browser. Keep that terminal running
and come back to it throughout the notebook — you'll see new runs appear
there live as you execute cells below.


## 1. Imports & dataset

In [1]:
import time
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay


Matplotlib is building the font cache; this may take a moment.


In [2]:

print("MLflow version:", mlflow.__version__)


MLflow version: 3.14.0


In [3]:
wine = load_wine(as_frame=True)
df = wine.frame
df["target_name"] = df["target"].map(dict(enumerate(wine.target_names)))
df.head()


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target,target_name
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0,class_0


In [4]:
X = df[wine.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Classes:", list(wine.target_names))


Train: (142, 13), Test: (36, 13)
Classes: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]


## 2. The problem, without MLflow

Here's how most people train and "track" a model before they adopt MLflow:
train it, print the metric, maybe jot it in a spreadsheet or a comment. Let's
do that for a couple of hyperparameter choices and feel the pain ourselves.


In [ ]:
# Attempt 1
model_v1 = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
model_v1.fit(X_train, y_train)
acc_v1 = accuracy_score(y_test, model_v1.predict(X_test))
print(f"n_estimators=100, max_depth=3  -> accuracy = {acc_v1:.4f}")

# Attempt 2 (changed max_depth... but did we change anything else? did we save attempt 1 anywhere?)
model_v2 = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
model_v2.fit(X_train, y_train)
acc_v2 = accuracy_score(y_test, model_v2.predict(X_test))
print(f"n_estimators=200, max_depth=None -> accuracy = {acc_v2:.4f}")

# Now imagine doing this 50 times across a week, with a colleague doing the same on their laptop.
# Which params gave which number? Where's the actual model file for the best one?


This "works," but it doesn't scale: nothing is saved automatically, nothing is
comparable side-by-side, and the trained model objects vanish the moment the
Python process ends. **This is exactly the gap MLflow Tracking fills.**


## 3. MLflow Tracking: your first run

Key ideas:
- An **experiment** is a named collection of runs (e.g. "wine-classification").
- A **run** is one execution of your training code. Every run gets its own ID,
  and can log:
  - **parameters** (`mlflow.log_param` / `log_params`) — hyperparameters, config
  - **metrics** (`mlflow.log_metric` / `log_metrics`) — accuracy, F1, loss, etc. (can be logged over time/steps too)
  - **artifacts** (`mlflow.log_artifact`) — any file: plots, CSVs, the trained model itself
- Everything happens inside a `with mlflow.start_run():` block.


In [ ]:
mlflow.set_tracking_uri("file:./mlruns")  # local folder; could also point at a remote tracking server
mlflow.set_experiment("wine-classification")


In [ ]:
with mlflow.start_run(run_name="rf_baseline") as run:
    n_estimators = 100
    max_depth = 3

    # 1. Log the hyperparameters
    mlflow.log_params({"n_estimators": n_estimators, "max_depth": max_depth})

    # 2. Train, exactly as before
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # 3. Log the metrics
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    mlflow.log_metrics({"accuracy": acc, "f1_macro": f1})

    # 4. Log a plot as an artifact
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=wine.target_names, ax=ax)
    ax.set_title("Confusion Matrix - rf_baseline")
    plt.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.show()

    # 5. Log the trained model itself, in MLflow's standard model format
    mlflow.sklearn.log_model(model, artifact_path="model", input_example=X_train.iloc[:5])

    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {acc:.4f} | F1 (macro): {f1:.4f}")


**Go check the MLflow UI now** (http://127.0.0.1:5000). Click into the
`wine-classification` experiment and you should see this run, its parameters,
metrics, the confusion matrix image, and the saved model — all captured
automatically, with zero manual note-taking.


## 4. Comparing multiple runs

The real payoff of tracking shows up once you have *many* runs to compare.
Let's sweep a few hyperparameter combinations, each as its own run, and then
pull all the results back into a single table with `mlflow.search_runs`.


In [ ]:
param_grid = [
    {"n_estimators": 100, "max_depth": 3},
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 200, "max_depth": 5},
    {"n_estimators": 300, "max_depth": None},
    {"n_estimators": 50,  "max_depth": 2},
]

for params in param_grid:
    with mlflow.start_run(run_name=f"rf_n{params['n_estimators']}_d{params['max_depth']}"):
        mlflow.log_params(params)

        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
        mlflow.log_metrics({"accuracy": acc, "f1_macro": f1})
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=X_train.iloc[:5])

print("Finished sweeping", len(param_grid), "runs.")


In [ ]:
# Pull every run from this experiment into a pandas DataFrame -- no manual bookkeeping needed
experiment = mlflow.get_experiment_by_name("wine-classification")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.accuracy DESC"])

# Show the columns that matter for comparison
cols = ["run_id", "tags.mlflow.runName", "params.n_estimators", "params.max_depth",
        "metrics.accuracy", "metrics.f1_macro"]
runs_df[cols].head(10)


In [ ]:
plt.figure(figsize=(8, 5))
plot_df = runs_df.sort_values("metrics.accuracy")
sns.barplot(data=plot_df, y="tags.mlflow.runName", x="metrics.accuracy", hue="tags.mlflow.runName",
            palette="viridis", legend=False)
plt.xlabel("Test Accuracy")
plt.ylabel("")
plt.title("All Tracked Runs, Compared")
plt.xlim(0.8, 1.01)
plt.tight_layout()
plt.show()


This is the core habit shift MLflow encourages: **every training attempt is a
run, and every run is automatically comparable** — no spreadsheet, no manual
copy-pasting of numbers, no "which script version did I use for this."


## 5. Autologging: even less boilerplate

For many popular libraries (scikit-learn, XGBoost, LightGBM, PyTorch,
TensorFlow/Keras, and more), MLflow can automatically log parameters, metrics,
and the model **without you writing a single `log_param` or `log_model`
call**. Turn it on once with `mlflow.sklearn.autolog()`.


In [ ]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="logreg_autolog"):
    # Just train the model like you normally would -- MLflow captures the rest
    clf = LogisticRegression(max_iter=2000, C=0.5)
    clf.fit(X_train, y_train)
    # Autologging captures params (C, max_iter, ...) and training-set metrics automatically.
    # It's still good practice to explicitly log your *test*-set metrics, since autolog
    # only sees what happens during .fit():
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    mlflow.log_metric("test_accuracy", test_acc)
    print(f"Test accuracy: {test_acc:.4f}")

# Turn autologging back off so the rest of this notebook stays explicit and easy to follow
mlflow.sklearn.autolog(disable=True)


Check the UI again — notice this run has extra auto-captured details (all of
`LogisticRegression`'s parameters, not just the ones you chose to log
manually) plus, for scikit-learn, things like a training scoring artifact.
Autologging is great for quick iteration; manual logging (Section 3) gives you
full control over exactly what gets recorded.


## 6. Registering the best model

Tracking answers "what happened in each run?" The **Model Registry** answers
the next question: "which model is *the* model we ship?" It gives every
registered model a name, a version history, and a lifecycle.

We'll find the best run programmatically, then register its model.


In [ ]:
best_run = runs_df.iloc[0]  # already sorted by accuracy DESC
best_run_id = best_run["run_id"]
print(f"Best run: {best_run['tags.mlflow.runName']} | accuracy={best_run['metrics.accuracy']:.4f} | run_id={best_run_id}")

model_uri = f"runs:/{best_run_id}/model"
registered_model_name = "WineClassifier"

registered = mlflow.register_model(model_uri=model_uri, name=registered_model_name)
print(f"Registered '{registered_model_name}' as version {registered.version}")


In the UI, click the **Models** tab — you'll see `WineClassifier` with version
1 pointing back at the run that produced it. If you register another run
later (e.g. after retraining on new data), it becomes version 2, version 3,
and so on — full history preserved.

### Promoting a model through its lifecycle

Modern MLflow (2.x) uses **aliases** (e.g. `@champion`, `@staging`) rather
than the older fixed "Staging/Production" stages, since aliases are more
flexible (you can define your own). If you're on an older MLflow version,
the equivalent call is `client.transition_model_version_stage(...)`.


In [ ]:
client = MlflowClient()

# Tag the newly registered version as the current "champion" model
client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=registered.version,
)
print(f"'{registered_model_name}' version {registered.version} is now aliased '@champion'")


## 7. Loading a registered model for inference

This is the payoff: anyone (a teammate, a deployment pipeline, a batch job)
can load *the exact model you registered* by name and alias — no need to know
which run produced it, or to have the original training code lying around.


In [ ]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@champion")

sample = X_test.iloc[:5]
predictions = loaded_model.predict(sample)

comparison = pd.DataFrame({
    "predicted": [wine.target_names[p] for p in predictions],
    "actual": [wine.target_names[a] for a in y_test.iloc[:5]],
})
comparison


## 8. Summary: what MLflow gave us that plain scikit-learn didn't

| Without MLflow | With MLflow |
|---|---|
| Metrics live in `print()` output or a notebook cell you might overwrite | Every metric is logged and queryable (`mlflow.search_runs`) |
| Hyperparameters are wherever the code happened to leave them | Every param is logged alongside its resulting metrics |
| The trained model object dies with the Python process | The model is saved in a standard format, versioned, and reloadable |
| "Which version is in production?" is answered by tribal knowledge | The Model Registry has one name (`WineClassifier`) with a full version history and lifecycle aliases |
| Comparing 10 experiments means opening 10 sets of notes | `mlflow.search_runs(...)` gives you one sortable table |

### Where to go next
- **MLflow Projects** — package training code (with its dependencies) so it runs reproducibly anywhere.
- **Model serving** — `mlflow models serve -m "models:/WineClassifier@champion"` spins up a local REST API for the model in one command.
- **Remote tracking servers** — point `mlflow.set_tracking_uri()` at a shared server (e.g. backed by a database and S3/Azure Blob storage) so a whole team logs to the same place.
- **`mlflow.evaluate()`** — a higher-level API that auto-generates a standard set of metrics and plots for a model, in a couple of lines.
